In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings("ignore")

In [5]:
# 1. CHART STYLING

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#2e3250',
    'axes.labelcolor':  '#c9d1e0',
    'axes.titlecolor':  '#ffffff',
    'xtick.color':      '#8b92a8',
    'ytick.color':      '#8b92a8',
    'text.color':       '#c9d1e0',
    'grid.color':       '#2e3250',
    'grid.linewidth':   0.6,
    'font.family':      'DejaVu Sans',
    'font.size':        10,
})
 
ACCENT = '#4f8ef7'
RED    = '#e05c5c'
GREEN  = '#4ecb8d'
YELLOW = '#f5a623'
PURPLE = '#a78bfa'

In [6]:
# 2. LOAD & CLEAN DATA

df = pd.read_csv('hotel_bookings.csv')
 
df['adr']      = pd.to_numeric(df['adr'], errors='coerce')
df['children'] = pd.to_numeric(df['children'], errors='coerce').fillna(0)
df = df[df['adr'] >= 0]  # drop invalid ADR rows
 
month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
df['arrival_date_month'] = pd.Categorical(
    df['arrival_date_month'], categories=month_order, ordered=True)
 
df['lead_time_bin'] = pd.cut(
    df['lead_time'],
    bins=[0, 30, 60, 90, 120, 180, 365, 737],
    labels=['0-30','31-60','61-90','91-120','121-180','181-365','365+'])
 
df['est_revenue'] = df['adr'] * (
    df['stays_in_weekend_nights'] + df['stays_in_week_nights'])

In [7]:
# 3. HELPERS


def style_ax(ax, title, xlabel='', ylabel=''):
    ax.set_facecolor('#1a1d27')
    ax.set_title(title, fontsize=11, fontweight='bold', color='white', pad=10)
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.grid(axis='y', alpha=0.4)
    for spine in ax.spines.values():
        spine.set_edgecolor('#2e3250')
 
def bar_labels(ax, fmt='{:.0f}%', fontsize=8):
    for p in ax.patches:
        h = p.get_height()
        if h > 0:
            ax.annotate(fmt.format(h),
                        (p.get_x() + p.get_width() / 2, h),
                        ha='center', va='bottom',
                        fontsize=fontsize, color='white')
 
def kpi_box(fig, x, y, w, h, value, label, color):
    ax = fig.add_axes([x, y, w, h])
    ax.set_facecolor(color + '22')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    for spine in ax.spines.values():
        spine.set_edgecolor(color); spine.set_linewidth(1.5)
    ax.set_xticks([]); ax.set_yticks([])
    ax.text(0.5, 0.65, value, ha='center', va='center',
            fontsize=22, fontweight='bold', color=color)
    ax.text(0.5, 0.20, label, ha='center', va='center',
            fontsize=9, color='#8b92a8')
 

In [8]:
# 4. SUMMARY STATISTICS
# -----------------------------------------------------------------------------
total       = len(df)
cancelled   = int(df['is_canceled'].sum())
cancel_rate = cancelled / total * 100
avg_adr     = df[df['is_canceled'] == 0]['adr'].mean()
rev_lost    = df[df['is_canceled'] == 1]['est_revenue'].sum()
lt_high     = df[df['lead_time'] > 120]['is_canceled'].mean() * 100
ota_rate    = df[df['market_segment'] == 'Online TA']['is_canceled'].mean() * 100
direct_rate = df[df['market_segment'] == 'Direct']['is_canceled'].mean() * 100
no_dep_rate = df[df['deposit_type'] == 'No Deposit']['is_canceled'].mean() * 100
 
print("=" * 55)
print("  HOTEL BOOKING CANCELLATION — KEY FINDINGS")
print("=" * 55)
print(f"  Total bookings          : {total:>10,}")
print(f"  Total cancellations     : {cancelled:>10,}")
print(f"  Overall cancel rate     : {cancel_rate:>9.1f}%")
print(f"  Avg daily rate (active) : €{avg_adr:>8.0f}")
print(f"  Estimated revenue lost  : €{rev_lost:>10,.0f}")
print(f"  Cancel rate >120d lead  : {lt_high:>9.1f}%")
print(f"  OTA cancel rate         : {ota_rate:>9.1f}%")
print(f"  Direct channel cancel   : {direct_rate:>9.1f}%")
print(f"  No-deposit cancel rate  : {no_dep_rate:>9.1f}%")
print("=" * 55)

  HOTEL BOOKING CANCELLATION — KEY FINDINGS
  Total bookings          :    119,389
  Total cancellations     :     44,224
  Overall cancel rate     :      37.0%
  Avg daily rate (active) : €     100
  Estimated revenue lost  : €16,727,237
  Cancel rate >120d lead  :      52.2%
  OTA cancel rate         :      36.7%
  Direct channel cancel   :      15.3%
  No-deposit cancel rate  :      28.4%


In [10]:
# PAGE 1 — OVERVIEW DASHBOARD
# =============================================================================
fig1, axes = plt.subplots(2, 3, figsize=(18, 10))
fig1.patch.set_facecolor('#0f1117')
fig1.suptitle('Hotel Booking Cancellation Analysis — Overview',
              fontsize=16, fontweight='bold', color='white', y=1.01)
 
for ax in axes[0]:
    ax.set_visible(False)
 
kpis = [
    (f'{total:,}',          'Total Bookings',             ACCENT),
    (f'{cancel_rate:.1f}%', 'Overall Cancellation Rate',  RED),
    (f'€{avg_adr:.0f}',     'Avg Daily Rate (Active)',     GREEN),
]
for i, (val, lbl, col) in enumerate(kpis):
    kpi_box(fig1, 0.05 + i * 0.32, 0.72, 0.28, 0.12, val, lbl, col)
 
# Cancellation rate by hotel type
ax_a = axes[1][0]
hotel_cr = df.groupby('hotel')['is_canceled'].mean() * 100
ax_a.bar(hotel_cr.index, hotel_cr.values, color=[RED, ACCENT], width=0.5)
style_ax(ax_a, 'Cancellation Rate by Hotel Type', '', 'Rate (%)')
bar_labels(ax_a)
 
# Cancellation rate by market segment
ax_b = axes[1][1]
seg_cr = (df.groupby('market_segment')['is_canceled']
            .mean().sort_values(ascending=False) * 100)
ax_b.bar(seg_cr.index, seg_cr.values,
         color=[RED if v > 30 else ACCENT for v in seg_cr.values])
ax_b.set_xticklabels(seg_cr.index, rotation=35, ha='right', fontsize=8)
style_ax(ax_b, 'Cancellation Rate by Market Segment', '', 'Rate (%)')
 
# Cancellation rate by deposit type
ax_c = axes[1][2]
dep_cr = (df.groupby('deposit_type')['is_canceled']
            .mean().sort_values(ascending=False) * 100)
ax_c.bar(dep_cr.index, dep_cr.values,
         color=[RED if v > 50 else GREEN for v in dep_cr.values], width=0.5)
style_ax(ax_c, 'Cancellation Rate by Deposit Type', '', 'Rate (%)')
bar_labels(ax_c)
 
plt.tight_layout()
fig1.savefig('eda_page1_overview.png', dpi=150,
             bbox_inches='tight', facecolor='#0f1117')
plt.close()
print("\n[✓] Page 1 — Overview saved")
 


[✓] Page 1 — Overview saved


In [11]:
# PAGE 2 — LEAD TIME & SEASONALITY
# =============================================================================
fig2, axes2 = plt.subplots(2, 2, figsize=(16, 10))
fig2.patch.set_facecolor('#0f1117')
fig2.suptitle('Lead Time Impact & Seasonal Patterns',
              fontsize=15, fontweight='bold', color='white')
 
# Lead time vs cancellation rate (line chart)
ax = axes2[0][0]
lt_cr = df.groupby('lead_time_bin', observed=True)['is_canceled'].mean() * 100
ax.plot(lt_cr.index, lt_cr.values, color=RED,
        marker='o', linewidth=2.5, markersize=7, markerfacecolor='white')
ax.fill_between(range(len(lt_cr)), lt_cr.values, alpha=0.15, color=RED)
ax.set_xticks(range(len(lt_cr)))
ax.set_xticklabels(lt_cr.index, rotation=30)
ax.axhline(52, color=YELLOW, linestyle='--', linewidth=1, alpha=0.8)
ax.text(5.5, 53, '52% at >120 days', color=YELLOW, fontsize=8)
style_ax(ax, 'Cancellation Rate by Lead Time (days)',
         'Lead Time Bucket', 'Rate (%)')
 
# Monthly bookings vs cancellations
ax = axes2[0][1]
monthly = (df.groupby('arrival_date_month', observed=True)
             .agg(total=('is_canceled','count'),
                  cancelled=('is_canceled','sum'))
             .reset_index())
x = range(len(monthly))
ax.bar(x, monthly['total'],     color=ACCENT, alpha=0.7, label='Total')
ax.bar(x, monthly['cancelled'], color=RED,    alpha=0.9, label='Cancelled')
ax.set_xticks(list(x))
ax.set_xticklabels([m[:3] for m in monthly['arrival_date_month']], rotation=45)
ax.legend(facecolor='#1a1d27', edgecolor='#2e3250', labelcolor='white')
style_ax(ax, 'Monthly Bookings vs Cancellations', 'Month', 'Count')
 
# ADR distribution: cancelled vs not cancelled
ax = axes2[1][0]
ax.hist(df[df['is_canceled'] == 0]['adr'].clip(0, 400),
        bins=50, color=GREEN, alpha=0.7, label='Not Cancelled')
ax.hist(df[df['is_canceled'] == 1]['adr'].clip(0, 400),
        bins=50, color=RED,   alpha=0.7, label='Cancelled')
ax.legend(facecolor='#1a1d27', edgecolor='#2e3250', labelcolor='white')
style_ax(ax, 'ADR Distribution: Cancelled vs Not Cancelled',
         'Average Daily Rate (€)', 'Count')
 
# New guest vs repeat guest cancellation rate
ax = axes2[1][1]
rg_cr  = df.groupby('is_repeated_guest')['is_canceled'].mean() * 100
labels = ['New Guest', 'Repeat Guest']
bars   = ax.bar(labels, rg_cr.values, color=[RED, GREEN], width=0.4)
for bar, val in zip(bars, rg_cr.values):
    ax.text(bar.get_x() + bar.get_width() / 2, val / 2,
            f'{val:.1f}%', ha='center', va='center',
            fontsize=14, fontweight='bold', color='white')
style_ax(ax, 'Cancellation Rate: New vs Repeat Guests', '', 'Rate (%)')
 
plt.tight_layout()
fig2.savefig('eda_page2_leadtime.png', dpi=150,
             bbox_inches='tight', facecolor='#0f1117')
plt.close()
print("[✓] Page 2 — Lead Time & Seasonality saved")
 

[✓] Page 2 — Lead Time & Seasonality saved


In [12]:
# PAGE 3 — REVENUE IMPACT & RECOMMENDATIONS
# =============================================================================
fig3, axes3 = plt.subplots(2, 2, figsize=(16, 10))
fig3.patch.set_facecolor('#0f1117')
fig3.suptitle('Revenue Leakage & Business Recommendations',
              fontsize=15, fontweight='bold', color='white')
 
# Revenue lost by customer type
ax = axes3[0][0]
rev_by_type = (df[df['is_canceled'] == 1]
               .groupby('customer_type')['est_revenue']
               .sum().sort_values(ascending=True) / 1e6)
ax.barh(rev_by_type.index, rev_by_type.values, color=RED, alpha=0.85)
for i, v in enumerate(rev_by_type.values):
    ax.text(v + 0.05, i, f'€{v:.1f}M', va='center', color='white', fontsize=9)
style_ax(ax, 'Estimated Revenue Lost by Customer Type', 'Revenue Lost (€M)', '')
ax.grid(axis='x', alpha=0.4)
ax.grid(axis='y', alpha=0)
 
# Top 10 countries by cancellations
ax = axes3[0][1]
top_c = df[df['is_canceled'] == 1]['country'].value_counts().head(10)
ax.bar(top_c.index, top_c.values,
       color=[RED if i == 0 else ACCENT for i in range(len(top_c))])
ax.set_xticklabels(top_c.index, rotation=45, ha='right')
style_ax(ax, 'Top 10 Countries by Cancellations', 'Country Code', 'Cancellations')
 
# Special requests vs cancellation rate
ax = axes3[1][0]
sr_cr = df.groupby('total_of_special_requests')['is_canceled'].mean() * 100
ax.bar(sr_cr.index, sr_cr.values,
       color=[GREEN if v < 20 else YELLOW if v < 35 else RED for v in sr_cr.values])
ax.set_xticks(sr_cr.index)
style_ax(ax, 'Cancellation Rate by Number of Special Requests',
         'Special Requests Count', 'Rate (%)')
 
# Business recommendations panel
ax = axes3[1][1]
ax.set_facecolor('#1a1d27')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xticks([]); ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_edgecolor('#2e3250')
ax.set_title('Key Business Recommendations',
             fontsize=11, fontweight='bold', color='white', pad=10)
 
recs = [
    (0.93, YELLOW,    '1. Non-refundable deposit for bookings > 90 days'),
    (0.78, GREEN,     '2. 5% discount for Direct Bookings vs OTA'),
    (0.63, ACCENT,    '3. Waitlist system for Jul-Aug peak months'),
    (0.48, PURPLE,    '4. Re-engage guests with 0 special requests'),
    (0.33, RED,       '5. Flag risk: long lead + OTA + no deposit'),
    (0.16, '#06b6d4', '6. Retention priority for Transient segment'),
]
for y_pos, col, text in recs:
    ax.add_patch(mpatches.FancyBboxPatch(
        (0.03, y_pos - 0.07), 0.94, 0.11,
        boxstyle='round,pad=0.01',
        facecolor=col + '22', edgecolor=col, linewidth=1.2))
    ax.text(0.07, y_pos - 0.01, text,
            va='center', fontsize=8.5, color='white', fontweight='bold')
 
plt.tight_layout()
fig3.savefig('eda_page3_revenue.png', dpi=150,
             bbox_inches='tight', facecolor='#0f1117')
plt.close()
print("[✓] Page 3 — Revenue Impact saved")
print("\n[✓] All 3 chart pages exported successfully.")
print("    Upload PNG files to your GitHub /images folder.")


[✓] Page 3 — Revenue Impact saved

[✓] All 3 chart pages exported successfully.
    Upload PNG files to your GitHub /images folder.
